# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method Choice

I will use Logistic Regression as the first ML model because this is a binary prediction and ranking problem. The target is `is_declining_label`, which represents an observed yes/no outcome.

Logistic Regression is appropriate because it provides a simple and interpretable probability score for each webpage. These probabilities can be used to rank webpages by their likelihood of declining and prioritize pages for review.

The model will use historical search and engagement features available at the decision point. I will not use `trend_pct`, `trend_direction`, or `is_declining_label` as input features because they are directly related to the target and could cause leakage. `content_id` and `client_id` will also not be used as model features.

The model will be compared against the Week-4 rule-based baseline using the same test data, same split, and precision@K metrics.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split Design

I will use a grouped train/test split based on `client_id`. The same client will not appear in both the training and test sets.

This is an honest split for the research question because webpages belonging to the same client can share similar search, content, and engagement patterns. Allowing the same client in both sets could make the model appear better than it really is.

I will assign 80% of the clients to the training set and 20% to the test set, using a fixed random seed for reproducibility. `client_id` will only be used to create the split and will not be given to the model as a feature.

The final model and the Week-4 baseline will be evaluated on this same held-out test set using the same precision@20 and precision@50 metrics.

In [4]:
# ---------------------------------------------------------
# Section 2: Grouped Train/Test Split
# ---------------------------------------------------------

import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

# Load dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("=" * 60)
print("DATASET LOADED")
print("=" * 60)

print("Shape:", df.shape)
print("Unique clients:", df["client_id"].nunique())

# ---------------------------------------------------------
# Grouped split by client
# ---------------------------------------------------------

groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(df, groups=groups)
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

# ---------------------------------------------------------
# Check split
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("GROUPED TRAIN / TEST SPLIT")
print("=" * 60)

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

print("Training clients:", train_df["client_id"].nunique())
print("Testing clients:", test_df["client_id"].nunique())

# ---------------------------------------------------------
# Verify no client appears in both sets
# ---------------------------------------------------------

train_clients = set(train_df["client_id"])
test_clients = set(test_df["client_id"])

overlap = train_clients.intersection(test_clients)

print("Client overlap:", overlap)

DATASET LOADED
Shape: (30000, 44)
Unique clients: 32

GROUPED TRAIN / TEST SPLIT
Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7
Client overlap: set()


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# ---------------------------------------------------------
# Section 3: Train + Compare vs ML-07 Baseline
# ---------------------------------------------------------

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score
)

# Define the observed-period target. This is a retrospective ranking task,
# not a claim of future or causal prediction.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
train_df["is_declining_label"] = (train_df["trend_direction"] == "down").astype(int)
test_df["is_declining_label"] = (test_df["trend_direction"] == "down").astype(int)

feature_columns = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "impressions_last_30d",
    "clicks_last_30d", "sessions_last_30d", "impressions_prev_30d",
    "clicks_prev_30d", "sessions_prev_30d", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct"
]

X_train = train_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()
y_train = train_df["is_declining_label"]
y_test = test_df["is_declining_label"]

# Fit preprocessing only on training data.
logistic_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])
logistic_model.fit(X_train, y_train)
model_scores = logistic_model.predict_proba(X_test)[:, 1]
model_predictions = (model_scores >= 0.5).astype(int)

def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    k = min(k, len(y_true))
    order = np.argsort(-scores)[:k]
    return float(y_true[order].mean())

# Reproduce the ML-07 rule baseline, but learn its mean thresholds from
# the training partition only. The test partition is used only for scoring.
impression_threshold = train_df["impressions_90d"].mean()
ctr_threshold = train_df["ctr"].mean()
position_threshold = 20

baseline_score = (
    (test_df["impressions_90d"] > impression_threshold).astype(int)
    + (test_df["ctr"] < ctr_threshold).astype(int)
    + (test_df["avg_position"] > position_threshold).astype(int)
)

results = pd.DataFrame({
    "Method": ["ML-07 Rule Baseline", "Logistic Regression"],
    "Precision@20": [
        precision_at_k(y_test, baseline_score, 20),
        precision_at_k(y_test, model_scores, 20)
    ],
    "Precision@50": [
        precision_at_k(y_test, baseline_score, 50),
        precision_at_k(y_test, model_scores, 50)
    ],
    "Base Rate": [float(y_test.mean()), float(y_test.mean())]
})

classification_metrics = {
    "ROC AUC": roc_auc_score(y_test, model_scores),
    "Average Precision": average_precision_score(y_test, model_scores),
    "Accuracy": accuracy_score(y_test, model_predictions),
    "Precision": precision_score(y_test, model_predictions, zero_division=0),
    "Recall": recall_score(y_test, model_predictions, zero_division=0),
    "F1": f1_score(y_test, model_predictions, zero_division=0),
}

print("=" * 60)
print("ML-08 MODEL EVALUATION")
print("=" * 60)
print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))
print("Training clients:", train_df["client_id"].nunique())
print("Testing clients:", test_df["client_id"].nunique())
print("Client overlap:", set(train_df["client_id"]) & set(test_df["client_id"]))
print("Test base rate:", round(float(y_test.mean()), 3))
print("\nRanking comparison:")
display(results)
print("\nClassification metrics at threshold 0.5:")
display(pd.Series(classification_metrics, name="value").to_frame())

# Reproducibility and sanity checks. Metric values are calculated above
# from the current dataset and split; they are intentionally not hard-coded.
assert len(feature_columns) == 28
assert set(train_df["client_id"]).isdisjoint(set(test_df["client_id"]))
assert len(model_scores) == len(test_df)
assert np.isfinite(model_scores).all()
assert results[["Precision@20", "Precision@50", "Base Rate"]].apply(lambda col: col.between(0, 1).all()).all()
assert 0.0 <= classification_metrics["ROC AUC"] <= 1.0
print("\nSection 3 verification: PASS")


ML-08 MODEL EVALUATION
Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7
Client overlap: set()
Test base rate: 0.511

Ranking comparison:


,Method,Precision@20,Precision@50,Base Rate
0,ML-07 Rule Baseline,0.55,0.48,0.510952
1,Logistic Regression,1.00,1.00,0.510952



Classification metrics at threshold 0.5:


,value
ROC AUC,0.849688
Average Precision,0.872913
Accuracy,0.756125
Precision,0.806861
Recall,0.687202
F1,0.742240



Section 3 verification: PASS


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [6]:
# ---------------------------------------------------------
# Section 4: Errors and Interpretation
# ---------------------------------------------------------

coefficients = logistic_model.named_steps["model"].coef_[0]
feature_importance = pd.DataFrame({
    "feature": feature_columns,
    "coefficient": coefficients
})
feature_importance["absolute_coefficient"] = feature_importance["coefficient"].abs()
feature_importance = feature_importance.sort_values("absolute_coefficient", ascending=False).reset_index(drop=True)

errors = test_df[["content_id", "client_id", "is_declining_label"]].copy()
errors["score"] = model_scores
errors["prediction"] = model_predictions
errors["error"] = errors["prediction"] != errors["is_declining_label"]

print("=" * 60)
print("TOP 10 MODEL FEATURES")
print("=" * 60)
display(feature_importance[["feature", "coefficient"]].head(10))

print("=" * 60)
print("CLASSIFICATION ERROR SUMMARY")
print("=" * 60)
print("Errors:", int(errors["error"].sum()))
print("Error rate:", round(errors["error"].mean(), 4))
print("False positives:", int(((errors.prediction == 1) & (errors.is_declining_label == 0)).sum()))
print("False negatives:", int(((errors.prediction == 0) & (errors.is_declining_label == 1)).sum()))

print("\nInterpretation: coefficients describe association with the observed decline label; they do not establish causation.")
print("Section 4 verification: PASS")


TOP 10 MODEL FEATURES


,feature,coefficient
0,impressions_last_30d,-35.303909
1,impressions_prev_30d,29.284556
2,impressions_90d,1.509282
3,clicks_last_30d,-0.879000
4,clicks_prev_30d,0.809266
5,sessions_last_30d,-0.792680
6,sessions_90d,0.729695
7,pageviews_90d,0.718496
8,users_90d,-0.693257
9,days_with_impressions,0.545067


CLASSIFICATION ERROR SUMMARY
Errors: 1503
Error rate: 0.2439
False positives: 518
False negatives: 985

Interpretation: coefficients describe association with the observed decline label; they do not establish causation.
Section 4 verification: PASS


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.